[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AI-Hypercomputer/maxtext/blob/main/src/maxtext/examples/sft_multimodal_gemma3_demo.ipynb)

# Gemma3 Multimodal Supervised Fine Tuning (SFT) Demo

## Overview

This notebook demonstrates multimodal Supervised Fine-Tuning (SFT) in MaxText using Gemma3-4B as an example. By the end, you will have a fine-tuned model checkpoint in HuggingFace format.

**Workflow:**
1. **Download & convert** the Gemma3-4B checkpoint from HuggingFace to MaxText format.
2. **Fine-tune** the model with SFT on the [ChartQA](https://huggingface.co/datasets/ahmed-masry/ChartQA) dataset — a visual question-answering benchmark that requires understanding charts and figures.
3. **Export** the trained MaxText checkpoint back to HuggingFace format.

**Hardware:** Given the relatively small size of Gemma3-4B, this notebook runs on a **v4-8**, **v5p-8**, or **v6e-4** TPU VM.

## Prerequisites

Before running this notebook, make sure your environment is set up for the method you are using. Follow the [Run MaxText Python Notebooks on TPUs](https://maxtext.readthedocs.io/en/latest/guides/run_python_notebook.html) guide and complete all steps for your chosen method (Google Colab, VS Code, or Local Jupyter Lab) before proceeding.

If you run into issues, refer to the [Common Pitfalls & Debugging](https://maxtext.readthedocs.io/en/latest/guides/run_python_notebook.html#common-pitfalls-debugging) section of the guide.

In [ ]:
try:
  import google.colab
  print("Running the notebook on Google Colab")
  IN_COLAB = True
except ImportError:
    print("Running the notebook on Visual Studio or JupyterLab")
    IN_COLAB = False

## Installation: MaxText and Post training Dependencies

**Running the notebook on Visual Studio or JupyterLab**: Before proceeding, create a virtual environment and install the required post-training dependencies by following Option 3: Installing [tpu-post-train] in the [MaxText installation guide](https://maxtext.readthedocs.io/en/latest/install_maxtext.html#from-source). Once the environment is set up, ensure the notebook is running within it.

In [ ]:
if IN_COLAB:
    # Clone the MaxText repository
    !git clone https://github.com/AI-Hypercomputer/maxtext.git
    %cd maxtext

    # Install uv, a fast Python package installer
    !pip install uv
    
    # Install MaxText and post-training dependencies
    import os
    os.environ["UV_TORCH_BACKEND"]="cpu"
    !uv pip install -e .[tpu-post-train] --resolution=lowest
    !install_tpu_post_train_extra_deps

#### Session restart Instructions for Colab:

1. Navigate to the menu at the top of the screen.
2. Click on **Runtime**.
3. Select **Restart session** from the dropdown menu.

You will be asked to confirm the action in a pop-up dialog. Click on **Yes**.

## Environment Setup

In [ ]:
import datetime
import jax
import os
import subprocess
import sys
from maxtext.configs import pyconfig
from maxtext.utils.globals import MAXTEXT_PKG_DIR
from maxtext.trainers.post_train.sft import train_sft_native
from etils import epath

print(f"MaxText installation path: {MAXTEXT_PKG_DIR}")

In [ ]:
if not jax.distributed.is_initialized():
    jax.distributed.initialize()
print(f"JAX version: {jax.__version__}")
print(f"JAX devices: {jax.devices()}")

In [ ]:
if IN_COLAB:
    from huggingface_hub import notebook_login
    notebook_login()
else:
    from huggingface_hub import login
    login()

## Model Configurations

In [ ]:
MODEL_NAME = "gemma3-4b"
TOKENIZER_NAME = "google/gemma-3-4b-it"

BASE_OUTPUT_DIRECTORY = f"{MAXTEXT_PKG_DIR}/sft_multimodal_gemma3_output"

# set the path to the model checkpoint (including `/0/items`) or leave empty to download from HuggingFace
MODEL_CHECKPOINT_PATH = ""
if not MODEL_CHECKPOINT_PATH:
   MODEL_CHECKPOINT_PATH = f"{BASE_OUTPUT_DIRECTORY}/gemma3_checkpoint"
   print("Model checkpoint will be downloaded from HuggingFace at: ",  MODEL_CHECKPOINT_PATH)
   print("Set MODEL_CHECKPOINT_PATH if you do not wish to download the checkpoint.")

RUN_NAME = datetime.datetime.now().strftime("%Y-%m-%d-%H-%M-%S")

## Download Gemma3-4B Model Checkpoint from Hugging Face

In [ ]:
if not epath.Path(MODEL_CHECKPOINT_PATH).exists():
    print("Converting checkpoint from HuggingFace...")
    env = os.environ.copy()
    env["JAX_PLATFORMS"] = "cpu"

    subprocess.run(
        [
            sys.executable,
            "-m", "maxtext.checkpoint_conversion.to_maxtext",
            f"{MAXTEXT_PKG_DIR}/configs/base.yml",
            f"model_name={MODEL_NAME}",
            f"base_output_directory={MODEL_CHECKPOINT_PATH}",
            "use_multimodal=True",
            "scan_layers=True",
            "skip_jax_distributed_system=True",
            "--eager_load_method=transformers",
            "--lazy_load_tensors=False",
        ],
        check=True,
        env=env
    )

    # The HF model cache is no longer needed after conversion to MaxText format.
    import shutil
    hf_cache = epath.Path(os.path.expanduser("~")) / ".cache" / "huggingface" / "hub"
    if hf_cache.exists():
        shutil.rmtree(str(hf_cache))

    MODEL_CHECKPOINT_PATH = os.path.join(MODEL_CHECKPOINT_PATH, "0/items")
else:
    print(f"Model checkpoint exists at {MODEL_CHECKPOINT_PATH}")

## MaxText configurations

In [ ]:
# Load configuration for SFT training
config_argv = [
    "",
    f"{MAXTEXT_PKG_DIR}/configs/post_train/sft-vision-chartqa.yml",
    f"load_parameters_path={MODEL_CHECKPOINT_PATH}",
    f"model_name={MODEL_NAME}",
    f"tokenizer_path={TOKENIZER_NAME}",
    "steps=10",
    "attention=dot_product",
    "per_device_batch_size=1",
    "max_prefill_predict_length=1024",
    "max_target_length=2048",
    "max_num_images_per_example=1",
    "scan_layers=True",
    f"base_output_directory={BASE_OUTPUT_DIRECTORY}",
    f"run_name={RUN_NAME}",
    "save_checkpoint_on_completion=True",
]

config = pyconfig.initialize_pydantic(config_argv)

print("✓ SFT configuration loaded:")
print(f" Model: {config.model_name}")
print(f" Training Steps: {config.steps}")
print("Model Checkpoint Path: ", config.load_parameters_path)
print(f" Output Directory: {config.base_output_directory}")

## SFT Training

In [ ]:
from absl import flags
if not flags.FLAGS.is_parsed():
    flags.FLAGS.mark_as_parsed()

import traceback

print("=" * 60)
print("🚀 Starting SFT Training...")
print("=" * 60)

try:
    _ = train_sft_native.train_loop(config, recorder=None)
    print("\n" + "=" * 60)
    print("✅ Training Completed Successfully!")
    print("=" * 60)
    print(f"📁 Checkpoints saved to: {config.checkpoint_dir}")
except Exception:
    print("\n" + "=" * 60)
    print("❌Training Failed!")
    print("=" * 60)
    traceback.print_exc()
    print("\nFor troubleshooting, refer to the Common Pitfalls & Debugging section:")
    print("https://maxtext.readthedocs.io/en/latest/guides/run_python_notebook.html#common-pitfalls-debugging")
    sys.exit(1)

## Convert MaxText Checkpoint to Hugging Face Format

In [ ]:
# Define the output directory for the Hugging Face checkpoint
hf_output_directory = epath.Path(BASE_OUTPUT_DIRECTORY) / "hf_checkpoint"

# Find the latest MaxText checkpoint
checkpoint_dir = epath.Path(config.checkpoint_dir)
step_dirs = [d.name for d in checkpoint_dir.iterdir() if d.name.isdigit() and d.is_dir()]
if not step_dirs:
    print(f"No checkpoint found in {checkpoint_dir}")
else:
    latest_step = max(step_dirs, key=int)
    maxtext_checkpoint_path = checkpoint_dir / latest_step / "items"

    print(f"Converting MaxText checkpoint from: {maxtext_checkpoint_path}")
    print(f"Saving Hugging Face checkpoint to: {hf_output_directory}")

    # Run the conversion script
    env = os.environ.copy()
    env["JAX_PLATFORMS"] = "cpu"

    subprocess.run(
        [
            sys.executable,
            "-m", "maxtext.checkpoint_conversion.to_huggingface",
            f"{MAXTEXT_PKG_DIR}/configs/base.yml",
            f"model_name={MODEL_NAME}",
            f"load_parameters_path={str(maxtext_checkpoint_path)}",
            f"base_output_directory={str(hf_output_directory)}",
            f"scan_layers={config.scan_layers}",
            "use_multimodal=true",
            "skip_jax_distributed_system=True",
            "weight_dtype=bfloat16",
        ],
        check=True,
        env=env
    )

    print("✓ Conversion completed successfully!")